In [ ]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TO CREATE MODEL
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# TO IGNORE WARNINGS
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# LOAD CSV
df = pd.read_csv('RealEstate_HousePrice_Dataset_4200.csv')

df.head()

In [ ]:
# Task 7: Independent AND Dependent variables identify.

# Dependent Variable (Y) 
# Independent Variables (X) 

X = df.drop(['house_price_inr', 'house_id'], axis=1)
y = df['house_price_inr']

print("Independent Variables (Features):")
print(X.columns.tolist())
print("\nDependent Variable (Target): house_price_inr")

In [ ]:
# Task 8: TO SEE WHICH FEATURE AFFECTS THE PRICE

# Calculate the correlation
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Which Feature is Related to Price? (Correlation Matrix)")
plt.show()

# MAKE A SCATTER PLOT
plt.figure(figsize=(8, 5))
plt.scatter(df['area_sqft'], df['house_price_inr'], color='purple', alpha=0.4)
plt.xlabel("Area (sqft)")
plt.ylabel("Price (INR)")
plt.title("Area vs Price - What is the Relationship?")
plt.show()

In [ ]:
# Task 9: PUT DATA TO TRAINING AND TESTING.

# 80% data IN training , 20% IN testing .

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data: {X_train.shape[0]} rows")
print(f"Testing data: {X_test.shape[0]} rows")

In [ ]:
# Task 10: Simple Linear Regression USING ONLY ONE FEATURE

X_train_simple = X_train[['area_sqft']]
X_test_simple = X_test[['area_sqft']]

# CREATE MODEL
slr_model = LinearRegression()
slr_model.fit(X_train_simple, y_train)

y_pred_simple = slr_model.predict(X_test_simple)

# Plotting the Regression Line
plt.figure(figsize=(8, 5))
plt.scatter(X_test_simple, y_test, color='blue', alpha=0.3, label='Actual Price')
plt.plot(X_test_simple, y_pred_simple, color='red', linewidth=2, label='Regression Line')
plt.xlabel("Area (sqft)")
plt.ylabel("Price (INR)")
plt.title("Simple Linear Regression: Area vs Price")
plt.legend()
plt.show()

# Print Slope and Intercept
print(f"Slope (m): {slr_model.coef_[0]:.2f}")
print(f"Intercept (c): {slr_model.intercept_:.2f}")

# Task 12: Check Assumptions (Validation)
residuals = y_test - y_pred_simple
plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True, color='green')
plt.title("Distribution of Residuals (Should be Normal)")
plt.show()

In [ ]:
# Task 13: Calculate Metrics

mse = mean_squared_error(y_test, y_pred_simple)
mae = mean_absolute_error(y_test, y_pred_simple)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_simple)

n = len(y_test)
p = 1 # ONLY USING 1 feature
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print("--- Simple Linear Regression Results ---")
print(f"MSE: {mse:,.2f}")
print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2 Score: {r2:.4f}")
print(f"Adjusted R2: {adj_r2:.4f}")

In [ ]:
# Task 15: NOW USE ALL FEATURES. 

multi_model = LinearRegression()
multi_model.fit(X_train, y_train)

y_pred_multi = multi_model.predict(X_test)

# Task 16: Compare Simple vs Multiple
r2_multi = r2_score(y_test, y_pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y_test, y_pred_multi))

print("--- Comparison ---")
print(f"Simple Regression R2: {r2:.4f}")
print(f"Multiple Regression R2: {r2_multi:.4f}")
print(f"Simple RMSE: {rmse:,.2f}")
print(f"Multiple RMSE: {rmse_multi:,.2f}")

In [ ]:
# Task 18: Try Polynomial Regression. START FROM DEGREE 2.

from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
y_pred_poly = poly_model.predict(X_test_poly)

r2_poly = r2_score(y_test, y_pred_poly)
print(f"Polynomial (Degree 2) R2: {r2_poly:.4f}")

if r2_poly > r2_multi:
    print("Polynomial is slightly better, but we need to check if it is overfitting.")
else:
    print("Polynomial result is worse than Multiple Regression. It might be overfitting.")

# Task 20: Check Overfitting/Underfitting Visually
poly_simple = PolynomialFeatures(degree=3)
X_train_area_poly = poly_simple.fit_transform(X_train[['area_sqft']])
X_test_area_poly = poly_simple.transform(X_test[['area_sqft']])

poly_viz = LinearRegression()
poly_viz.fit(X_train_area_poly, y_train)

# Graph
plt.figure(figsize=(10, 6))
plt.scatter(X_test[['area_sqft']], y_test, color='gray', alpha=0.3, label='Actual')

sorted_idx = X_test[['area_sqft']].sort_values(by='area_sqft').index
plt.plot(X_test[['area_sqft']].loc[sorted_idx], 
         poly_viz.predict(X_test_area_poly)[np.argsort(X_test[['area_sqft']].values.flatten())], 
         color='red', linewidth=2, label='Polynomial Curve (Degree 3)')
plt.title("Polynomial Regression - Overfitting Check")
plt.legend()
plt.show()

In [ ]:
# Task 22: Batch Gradient Descent from Scratch

# First, we need to scale the data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[['area_sqft']])
y_train_arr = y_train.values.reshape(-1, 1)

def batch_gd(X, y, lr=0.01, epochs=100):
    m = len(y)
    theta = 0 # weight
    bias = 0
    cost_list = []
    
    for i in range(epochs):
        y_pred = theta * X + bias
        error = y_pred - y
        
        # Calculate Derivatives (Gradients)
        d_theta = (2/m) * np.sum(X * error)
        d_bias = (2/m) * np.sum(error)
        
        # Update the weights
        theta = theta - lr * d_theta
        bias = bias - lr * d_bias
        
        cost = np.mean(error**2)
        cost_list.append(cost)
        
    return theta, bias, cost_list

theta_bgd, bias_bgd, cost_bgd = batch_gd(X_train_scaled, y_train_arr, lr=0.01, epochs=100)


# Task 23: Stochastic Gradient Descent (SGD)

def sgd(X, y, lr=0.01, epochs=50):
    m = len(y)
    theta = 0
    bias = 0
    
    for epoch in range(epochs):
        for i in range(m):
            xi = X[i]
            yi = y[i]
            
            y_pred = theta * xi + bias
            error = y_pred - yi
            
            d_theta = 2 * xi * error
            d_bias = 2 * error
            
            theta = theta - lr * d_theta
            bias = bias - lr * d_bias
            
    return theta, bias

theta_sgd, bias_sgd = sgd(X_train_scaled, y_train_arr, lr=0.01, epochs=50)


# Task 24: Mini-Batch Gradient Descent

def mini_batch_gd(X, y, lr=0.01, epochs=50, batch_size=32):
    m = len(y)
    theta = 0
    bias = 0
    
    for epoch in range(epochs):
        indices = np.random.permutation(m)
        X_shuffled = X[indices]
        y_shuffled = y[indices]
        
        for i in range(0, m, batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]
            
            y_pred = theta * X_batch + bias
            error = y_pred - y_batch
            
            d_theta = (2/len(X_batch)) * np.sum(X_batch * error)
            d_bias = (2/len(X_batch)) * np.sum(error)
            
            theta = theta - lr * d_theta
            bias = bias - lr * d_bias
            
    return theta, bias

theta_mbgd, bias_mbgd = mini_batch_gd(X_train_scaled, y_train_arr, lr=0.01, epochs=50)


# Task 25: Compare Convergence

print("Batch GD Theta:", theta_bgd)
print("SGD Theta:", theta_sgd)
print("Mini-Batch GD Theta:", theta_mbgd)

# Plot the cost
plt.plot(cost_bgd, label='Batch GD')
plt.title("Batch GD Convergence (Cost is Decreasing)")
plt.xlabel("Epochs")
plt.ylabel("Cost (MSE)")
plt.legend()
plt.show()

In [ ]:
# Task 29: Final Report

print("--- FINAL REPORT ---")
print("1. Best Model: Multiple Linear Regression")
print("   Reason: It used all features (area, location, bedrooms) and gave a stable R2 score.")
print("2. Impact of Gradient Descent: I learned that Batch GD converges smoothly,")
print("   while SGD is faster but noisy.")
print("3. Overfitting/Underfitting: The simple model was underfitting, and the Polynomial model was overfitting.")
print("4. Business Interpretation: The company should buy houses with bigger areas and better location scores,")
print("   because these factors drive the price up the most.")

# Task 30: Submission
print("\nProject Complete! Ready to submit.")